# 🧹 Lesson 2.5 — Data Cleaning and Preprocessing

**AI Crash Course for Absolute Beginners**

Raw data is almost never ready for an AI model. In this notebook you will:
- Find and handle missing values
- Detect and deal with outliers
- Encode categorical variables
- Scale numerical features
- Split data into train and test sets

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Build a realistic messy dataset
np.random.seed(42)
n = 200

df = pd.DataFrame({
    "age"         : np.random.randint(18, 70, n).astype(float),
    "income"      : np.random.normal(55000, 15000, n),
    "education"   : np.random.choice(["high_school", "bachelor", "master", "phd"], n),
    "hours_per_week": np.random.randint(20, 60, n).astype(float),
    "purchased"   : np.random.choice([0, 1], n, p=[0.6, 0.4])
})

# Inject messiness
df.loc[np.random.choice(n, 20, replace=False), "age"]          = np.nan
df.loc[np.random.choice(n, 15, replace=False), "income"]       = np.nan
df.loc[np.random.choice(n, 10, replace=False), "education"]    = np.nan
df.loc[np.random.choice(n,  5, replace=False), "hours_per_week"] = 999  # outlier
df.loc[np.random.choice(n,  3, replace=False), "income"]       = -99999 # outlier

print(f"Dataset shape: {df.shape}")
df.head(8)

---
## Step 1 — Audit: Find the Problems

In [ ]:
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing: {df.isnull().sum().sum()} out of {df.size} cells")
print(f"Missing rate : {df.isnull().mean().mean():.1%}")

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Summary Statistics ===")
df.describe().round(1)

---
## Step 2 — Outlier Detection and Removal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))

axes[0].boxplot(df["hours_per_week"].dropna())
axes[0].set_title("Hours/Week — raw (note outlier at 999)")

axes[1].boxplot(df["income"].dropna())
axes[1].set_title("Income — raw (note -99999 outlier)")

plt.tight_layout()
plt.show()

In [ ]:
# IQR method: flag values beyond 1.5 * inter-quartile range
def remove_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_mask = (series < lower) | (series > upper)
    print(f"  Outliers found: {outlier_mask.sum()}  (range {lower:.0f} to {upper:.0f})")
    return series.where(~outlier_mask, other=np.nan)  # replace outliers with NaN

print("hours_per_week:")
df["hours_per_week"] = remove_outliers_iqr(df["hours_per_week"])

print("income:")
df["income"] = remove_outliers_iqr(df["income"])

---
## Step 3 — Handling Missing Values

In [ ]:
# Numerical: fill with median (robust to remaining skew)
for col in ["age", "income", "hours_per_week"]:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"  {col}: filled with median = {median_val:.1f}")

# Categorical: fill with most frequent value (mode)
mode_education = df["education"].mode()[0]
df["education"].fillna(mode_education, inplace=True)
print(f"  education: filled with mode = '{mode_education}'")

print(f"\nMissing values remaining: {df.isnull().sum().sum()}")

---
## Step 4 — Encode Categorical Variables

In [ ]:
# One-hot encoding — converts 'education' into binary columns
df_encoded = pd.get_dummies(df, columns=["education"], drop_first=True)

print("Columns after one-hot encoding:")
print(df_encoded.columns.tolist())
df_encoded.head(3)

In [ ]:
# Ordinal encoding — when there is a natural order
edu_order = {"high_school": 0, "bachelor": 1, "master": 2, "phd": 3}
df["education_ordinal"] = df["education"].map(edu_order)
print("Ordinal encoding sample:")
print(df[["education", "education_ordinal"]].drop_duplicates().sort_values("education_ordinal"))

---
## Step 5 — Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num_features = ["age", "income", "hours_per_week"]

# Standard scaling: mean=0, std=1
std_scaler = StandardScaler()
df_std = df.copy()
df_std[num_features] = std_scaler.fit_transform(df[num_features])

# Min-Max scaling: values between 0 and 1
mm_scaler = MinMaxScaler()
df_mm = df.copy()
df_mm[num_features] = mm_scaler.fit_transform(df[num_features])

print("=== StandardScaler results ===")
print(df_std[num_features].describe().round(2))

print("\n=== MinMaxScaler results ===")
print(df_mm[num_features].describe().round(2))

---
## Step 6 — Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_std[num_features]         # features
y = df["purchased"]              # target label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set  : {X_train.shape[0]} samples ({X_train.shape[0]/len(X):.0%})")
print(f"Test set      : {X_test.shape[0]} samples ({X_test.shape[0]/len(X):.0%})")
print(f"\nClass balance in train set:")
print(y_train.value_counts(normalize=True).round(2))
print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).round(2))

---
## The Full Preprocessing Pipeline

In production, wrap everything into a sklearn `Pipeline` so it applies consistently to new data:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# A pipeline for numerical features
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # fill missing
    ("scaler",  StandardScaler()),                  # then scale
])

# Demonstrate on a tiny array with a missing value
sample = np.array([[25, np.nan, 40],
                   [35, 60000, 38],
                   [45, 80000, 45]])

result = num_pipeline.fit_transform(sample)
print("Pipeline output (each column has mean≈0):")
print(result.round(3))

---
## Challenge Exercises

1. **Custom outlier rule**: Instead of IQR, cap income values to the 5th and 95th percentiles using `np.clip` and `np.percentile`.
2. **Missing visualisation**: Use `df.isnull()` with a heatmap (`sns.heatmap`) to visualise which rows have missing values.
3. **Pipeline extension**: Add `from sklearn.preprocessing import OneHotEncoder` and `ColumnTransformer` to handle both numerical and categorical columns in one pipeline.

---
## Summary

| Step | Technique | When to use |
|---|---|---|
| Find missing | `df.isnull().sum()` | Always first |
| Remove outliers | IQR, Z-score | Before imputation |
| Fill missing (num) | Median / mean | Numerical cols |
| Fill missing (cat) | Mode | Categorical cols |
| Encode categories | One-hot / ordinal | Before model training |
| Scale features | StandardScaler | Distance-based models |
| Split data | 80/20, stratified | Always last |

**Next lesson:** [Lesson 3.1 — What is Machine Learning?](https://GirlEf.github.io/ai-crash-course/PHASE%20THREE/LESSON%203.1.html)